# midi_gpt inference

Generate UK garage bass and arp MIDI from a text request.

The drum loop in `drum_samples/` supplies the kick pocket that the note models are
conditioned on; the request supplies key, mood, and length, which become a chord
progression. Output is symbolic — MIDI lands in `output/` for you to voice in a DAW.

This notebook runs the models in this kernel, so **the kernel must be the project
`.venv`**. The next cell checks that and tells you how to fix it if not.


In [ ]:
from pathlib import Path
import importlib
import sys


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "inference").is_dir() and (candidate / "utils").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from inside the midi_gpt project directory.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import librosa, mido, numpy, torch  # noqa: F401
except ModuleNotFoundError as error:
    raise RuntimeError(
        f"This kernel is missing {error.name!r}, so it is not the project environment.\n"
        f"Kernel python: {sys.executable}\n"
        f"Expected:      {PROJECT_ROOT / '.venv' / 'bin' / 'python'}\n\n"
        "Fix: run `uv sync` in the project, then select the .venv kernel "
        "(VS Code: Select Kernel -> Python Environments -> .venv)."
    ) from error

# Re-import the project from disk. A kernel that has been open across an edit keeps the
# version it first imported, so changes to the source would silently do nothing — which
# is exactly how a long-lived kernel keeps writing the old folder names. Order matters:
# leaves first and make_track last, so its `from ... import` bindings pick up the fresh
# objects rather than the stale ones.
from utils import config, audio_features, chords, midi_utils
from model import gpt_model, note_model
from inference import make_track
for _module in (config, audio_features, chords, midi_utils, gpt_model, note_model, make_track):
    importlib.reload(_module)

from utils.config import GRID_BARS, GRID_REF_BPM, NOTE_BARS, NOTE_STEPS

DRUM_DIR = PROJECT_ROOT / "drum_samples"
OUTPUT_DIR = PROJECT_ROOT / "output"

loops = sorted(DRUM_DIR.glob("*.wav"))
checkpoints = sorted((PROJECT_ROOT / "checkpoints").glob("*_notes_gpt*.pt"))

print(f"project    {PROJECT_ROOT}")
print(f"device     {make_track.pick_device()}")
print("drum loops in drum_samples/:")
for _i, _p in enumerate(loops):
    _b = make_track.detect_bpm(_p)
    print(f"  [{_i}] {_p.name:<34} {_b or 130:.0f} BPM{'' if _b else ' (assumed — put it in the filename)'}")
if not loops:
    print("  NONE — drop a .wav in drum_samples/")
import re as _re
_tags = sorted({_m.group(1) for _p in checkpoints
                if (_m := _re.search(r"_(ft\d+)\.pt$", _p.name))},
               key=lambda t: int(t[2:]))
print(f"weights available: base, {', '.join(_tags)}"
      f"   (latest = {make_track.latest_ckpt('bass').name.split('_')[-1][:-3]})")
print(f"output     {OUTPUT_DIR}")
# Proves the loaded code is current: a stale kernel shows a name without a timestamp.
print(f"folders    {make_track.run_folder(OUTPUT_DIR, 'example').name}")

## Configuration

Edit these, then run the rest of the notebook.


In [ ]:
# What to generate
REQUEST    = "8 bar UKG track in F minor, dark"
BPM        = None    # output tempo; None uses the genre's native 130
CANDIDATES = 10      # best-of-N per 4-bar section; higher is slower but picks better

# Drum loop — the rhythm skeleton. Pick by index from the table printed above, or by
# a substring of the filename ("house", "ukg"). None takes the first loop.
# The loop only supplies its kick pattern; its audio is never mixed in.
DRUM       = "ukg"
DRUM_BPM   = None    # None reads the tempo from the filename, else assumes 130

# Variation. SEED = None gives new music every run. Generation is deterministic, so
# a fixed seed reproduces one exactly — paste back the seed the run prints below.
SEED       = None

# Sampling temperature. None uses the mood preset (bass 1.2 / arp 1.3 for dark).
# A weak diversity lever: 1.2 -> 1.8 barely widens the note choice but drops
# kick-lock from 0.87 to 0.61. Vary SEED or DRUM for different music, not this.
BASS_TEMP  = None
ARP_TEMP   = None

# Chords. "fixed" repeats the mood template's plain triads every section.
# "color" recolors each chord with an in-key 7th or suspension, freshly per section,
# so bars 1-4 and 5-8 differ harmonically. Roots and major/minor never move, and a
# coloring is kept only if all its tones are diatonic. Seeded by SEED.
CHORDS     = "color"

# Overrides — leave as None to accept the defaults
GENRE      = None    # "ukg-dark", "ukg-classic", or None to read the mood from REQUEST

# Which weights. "latest" takes the highest ftN, "base" the untuned models, or name a
# checkpoint directly — "ft2", "ft3". Running the same SEED under two of these is a
# direct A/B: the conditioning is identical, so any difference is the weights.
WEIGHTS    = "latest"
NO_ARP     = False   # True generates bass only


def resolve_drum(choice):
    """Index, filename substring, or None -> a path in drum_samples/."""
    loops = make_track.drum_loops()
    if not loops:
        raise RuntimeError(f"No .wav in {DRUM_DIR}. Drop a drum loop there.")
    if choice is None:
        return loops[0]
    if isinstance(choice, int):
        return loops[choice]
    matches = [p for p in loops if str(choice).lower() in p.name.lower()]
    if not matches:
        raise RuntimeError(
            f"No loop matching {choice!r}. Available: {[p.name for p in loops]}"
        )
    return matches[0]


DRUM_PATH = resolve_drum(DRUM)
print(f"drum loop -> {DRUM_PATH.name}"
      f"   ({make_track.detect_bpm(DRUM_PATH) or 130:.0f} BPM)")

## Run

Calls the same code path as `python make_track.py`, in this process.


In [ ]:
import time

argv = ["--request", REQUEST, "--candidates", str(CANDIDATES)]
if SEED is not None:
    argv += ["--seed", str(SEED)]          # omitted -> make_track picks a fresh one
if BASS_TEMP is not None:
    argv += ["--bass-temp", str(BASS_TEMP)]
if ARP_TEMP is not None:
    argv += ["--arp-temp", str(ARP_TEMP)]
if BPM is not None:
    argv += ["--bpm", str(BPM)]
if GENRE is not None:
    argv += ["--genre", GENRE]
argv += ["--drum", str(DRUM_PATH), "--chords", CHORDS]
if DRUM_BPM is not None:
    argv += ["--drum-bpm", str(DRUM_BPM)]
if WEIGHTS == "base":
    argv += ["--base"]
elif WEIGHTS != "latest":
    for _role in ("bass", "arp"):
        _ckpt = PROJECT_ROOT / "checkpoints" / f"{_role}_notes_gpt_{WEIGHTS}.pt"
        if not _ckpt.exists():
            raise RuntimeError(f"No such checkpoint: {_ckpt.name}. "
                               f'Set WEIGHTS to "latest", "base", or an existing ftN.')
        argv += [f"--{_role}-ckpt", str(_ckpt)]
if NO_ARP:
    argv += ["--no-arp"]

started = time.time()
RUN = make_track.main(argv)
print(f"\nrun folder: {RUN.name}   ({time.time() - started:.1f}s)")

## Validation scores

`chord_fit` is the share of generated notes that land on a tone of the current chord.
`kick_lock` is the correlation between note onsets and the kick grid — the number that
says whether the rhythm conditioning actually took.


In [ ]:
import json

metadata = json.loads((RUN / "metadata.json").read_text())

print(f"{metadata['progression']}   {metadata['bpm']:.0f} BPM   "
      f"{metadata['parsed']['bars']} bars   {metadata['sections']} section(s)")
print(f"drum loop: {metadata['drum_loop']}\n")
print(f"{'role':<6} {'chord_fit':>10} {'kick_lock':>10} {'notes':>7}   model")
for role, scores in metadata["validation"].items():
    print(f"{role:<6} {scores['chord_fit']:>10.3f} {scores['kick_lock']:>10.3f} "
          f"{scores['notes']:>7}   {metadata['models'][role]}")

## What was generated

The kick grid is the conditioning input; the note rows are what the models produced
against it. Onsets sitting in the same column as a kick are what `kick_lock` measures.


In [ ]:
import mido
import numpy as np
from IPython.display import HTML, display

import librosa
from utils.audio_features import onset_grid
from utils.config import GRID_STEPS, SAMPLE_RATE

NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
KICK, PITCH, SUSTAIN = "#B4671A", "#35459A", "rgba(53,69,154,0.22)"


def note_name(midi):
    return f"{NAMES[midi % 12]}{midi // 12 - 1}"


def kick_grid(path, n_steps):
    """Rebuild exactly the grid make_track fed the model."""
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    # same alignment make_track applies, or the grid drawn here would not
    # match the one the model was conditioned on
    audio = make_track.align_to_grid_tempo(audio, metadata.get("drum_bpm", 130))
    envelope = onset_grid(audio)
    binned = np.array([b.max() if b.size else 0.0
                       for b in np.array_split(envelope, GRID_STEPS)])
    grid = (binned > 0.25).astype(np.float32)
    return np.tile(grid, int(np.ceil(n_steps / len(grid))))[:n_steps]


def midi_steps(path, n_steps):
    """MIDI back to per-step tokens: (pitch | 'sus' | None) for each sixteenth."""
    midi_file = mido.MidiFile(path)
    ticks_per_step = midi_file.ticks_per_beat // 4
    tick, held, notes = 0, {}, []
    for message in midi_file.tracks[0]:
        tick += message.time
        if message.type == "note_on" and message.velocity > 0:
            held[message.note] = tick
        elif message.type == "note_off" or (message.type == "note_on" and not message.velocity):
            if message.note in held:
                start = held.pop(message.note)
                notes.append((message.note, start // ticks_per_step,
                              max(1, (tick - start) // ticks_per_step)))
    row = [None] * n_steps
    for pitch, start, duration in notes:
        if start < n_steps:
            row[start] = pitch
            for offset in range(start + 1, min(start + duration, n_steps)):
                row[offset] = "sus"
    return row


def render(rows, n_steps, per_bar=16):
    cell = ("display:inline-block;width:26px;height:22px;line-height:22px;"
            "text-align:center;font:600 9px ui-monospace,Menlo,monospace;"
            "box-sizing:border-box;border:1px solid rgba(128,128,128,0.28);")
    html = ['<div style="overflow-x:auto;padding:4px 0">']
    for label, row in rows:
        html.append('<div style="white-space:nowrap;margin-bottom:2px">')
        html.append(f'<span style="display:inline-block;width:80px;'
                    f'font:600 10px ui-monospace,Menlo,monospace;opacity:0.7">{label}</span>')
        for step in range(n_steps):
            edge = "border-left:2px solid rgba(128,128,128,0.6);" if step % per_bar == 0 else ""
            value = row[step]
            if value is None:
                html.append(f'<span style="{cell}{edge}"></span>')
            elif value == "kick":
                html.append(f'<span style="{cell}{edge}background:{KICK};color:#fff">K</span>')
            elif value == "sus":
                html.append(f'<span style="{cell}{edge}background:{SUSTAIN}"></span>')
            else:
                html.append(f'<span style="{cell}{edge}background:{PITCH};color:#fff">'
                            f'{note_name(value)}</span>')
        html.append("</div>")
    html.append("</div>")
    return "".join(html)


drum_path = PROJECT_ROOT / "drum_samples" / metadata["drum_loop"]
total_steps = metadata["parsed"]["bars"] * 16
grid = kick_grid(drum_path, total_steps)

rows = [("kick", ["kick" if v else None for v in grid])]
for role in metadata["validation"]:
    rows.append((role, midi_steps(RUN / "stems" / f"{role}.mid", total_steps)))

for section in range(metadata["sections"]):
    lo, hi = section * NOTE_STEPS, (section + 1) * NOTE_STEPS
    display(HTML(f'<div style="font:600 11px ui-monospace,Menlo,monospace;'
                 f'opacity:0.7;margin-top:10px">section {section + 1} '
                 f'&mdash; bars {section * NOTE_BARS + 1}&ndash;{(section + 1) * NOTE_BARS}</div>'))
    display(HTML(render([(label, row[lo:hi]) for label, row in rows], hi - lo)))

## Notes written

Read back from the `.mid` files themselves, so this is exactly what `write_midi`
emitted — not the tokens it was given. Position is `bar.beat.sixteenth`, 1-indexed
the way a DAW shows it.


In [ ]:
def midi_notes(path):
    """Read a written .mid back into (midi, start_step, duration_steps), in time order."""
    midi_file = mido.MidiFile(path)
    ticks_per_step = midi_file.ticks_per_beat // 4
    tick, held, notes = 0, {}, []
    for message in midi_file.tracks[0]:
        tick += message.time
        if message.type == "note_on" and message.velocity > 0:
            held[message.note] = tick
        elif message.type == "note_off" or (message.type == "note_on" and not message.velocity):
            if message.note in held:
                start = held.pop(message.note)
                notes.append((message.note, start // ticks_per_step,
                              max(1, (tick - start) // ticks_per_step)))
    return sorted(notes, key=lambda n: (n[1], n[0]))


def position(step):
    """Step index -> 'bar.beat.sixteenth', 1-indexed like a DAW ruler."""
    return f"{step // 16 + 1}.{step % 16 // 4 + 1}.{step % 4 + 1}"


for role in metadata["validation"]:
    notes = midi_notes(RUN / "stems" / f"{role}.mid")
    gaps = [b - (a + d) for (_, a, d), (_, b, _) in zip(notes, notes[1:])]
    held = sum(d for _, _, d in notes)
    span = notes[-1][1] + notes[-1][2] if notes else 1

    print(f"\n{role}.mid — {len(notes)} notes, {metadata['parsed']['bars']} bars @ "
          f"{metadata['bpm']:.0f}  ({held / span:.0%} sounding)")
    print(f"  {'pos':<9} {'note':<5} {'midi':>4} {'step':>5} {'len':>4} {'beats':>6}  vel")
    for pitch, start, duration in notes:
        print(f"  {position(start):<9} {note_name(pitch):<5} {pitch:>4} {start:>5} "
              f"{duration:>4} {duration / 4:>6.2f}   90")
    if gaps:
        silent = sum(g for g in gaps if g > 0)
        print(f"  {sum(1 for g in gaps if g > 0)}/{len(gaps)} notes followed by a gap; "
              f"{silent} silent steps between notes")

## Files

Load these into your DAW. Every run writes a new timestamped folder under
`output/`, so nothing here is overwritten by the next one.


In [ ]:
for path in sorted(RUN.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size:>8,} B  {path.relative_to(OUTPUT_DIR)}")

runs = sorted((p for p in OUTPUT_DIR.iterdir() if p.is_dir()), reverse=True)
print(f"\n{len(runs)} run(s) in output/ — most recent first:")
for path in runs[:8]:
    print(f"  {'->' if path == RUN else '  '} {path.name}")